# Stage 4 — Compound-shift-aware checkpoint selection

**Hypothesis being tested**: the attention gate failed to learn to favor Grammatical
features (the one category proven robust under compound shift) because checkpoint
selection used `val` (reddit domain, train generators) — an in-domain, in-generator
signal that every feature category already handles near-perfectly, giving zero gradient
pressure toward the generalization-robust category.

**Fix tested here**: add a genuinely disjoint validation signal — `val_compound`
(reddit domain + **dolly**, the held-out generator) — that the model has never seen
during training. Verified against real data before building: reddit has 2,000 spare
human documents (val already used 1,000 of 3,000) and 3,000 dolly-generated documents
in reddit, none touched by train/val/testA/testB/testC. Domain stays reddit deliberately
(not peerread) so testA/testC remain completely untouched and comparable to the
already-audited results.

**Reuses everything already computed** (`m4_stage2.pkl`: splits, stylometric features,
category dims) rather than rebuilding from scratch. Only new work: extract features +
embeddings for the ~600-sample `val_compound` set, then retrain the attention-gated
Hybrid with checkpoint selection swapped to use it, and compare against the original
(already-audited) Hybrid results via McNemar.

## 0. Force a clean rerun (only needed if you have run this notebook before)

Checkpointing means a previous run's per-seed results are reused automatically if they
exist on disk. That is normally the correct behavior, but it also means fixing a bug in
this notebook and re-running it does NOT produce new results unless the stale checkpoints
from the buggy run are also removed. Run the cell below once if you have executed this
notebook before (harmless no-op on a first-ever run).

In [ ]:
import glob, os

# Points at the checkpoint directory used throughout this notebook.
_CKPT_DIR_PRECHECK = '/kaggle/working/checkpoints'
stale = glob.glob(f'{_CKPT_DIR_PRECHECK}/m4_hybrid_v2_compoundval_seed*.pkl')
if stale:
    print(f'Removing {len(stale)} checkpoint(s) from a previous run of THIS experiment:')
    for f in stale:
        print(' ', f)
        os.remove(f)
else:
    print('No previous checkpoints found for this experiment -- nothing to clean.')


## 1. Setup + restore checkpoints from previous stages

In [ ]:
!pip install -q transformers accelerate scikit-learn pandas numpy nltk tqdm matplotlib shap statsmodels

import os, re, json, random, warnings, math, pickle, shutil, glob
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
from nltk.corpus import stopwords

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, GPT2LMHeadModel, get_cosine_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
ARTIFACTS_DIR = '/kaggle/working/artifacts'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Restore from a previous session's committed output, if any (same pattern as prior stages)
restored = 0
for candidate in glob.glob('/kaggle/input/*/checkpoints') + glob.glob('/kaggle/input/*/*/checkpoints'):
    if os.path.isdir(candidate) and os.path.abspath(candidate) != os.path.abspath(CHECKPOINT_DIR):
        for item in os.listdir(candidate):
            dst = os.path.join(CHECKPOINT_DIR, item)
            if not os.path.exists(dst):
                src_path = os.path.join(candidate, item)
                (shutil.copytree if os.path.isdir(src_path) else shutil.copy2)(src_path, dst)
                restored += 1
        break
for candidate in glob.glob('/kaggle/input/*/artifacts') + glob.glob('/kaggle/input/*/*/artifacts'):
    if os.path.isdir(candidate) and os.path.abspath(candidate) != os.path.abspath(ARTIFACTS_DIR):
        for item in os.listdir(candidate):
            dst = os.path.join(ARTIFACTS_DIR, item)
            if not os.path.exists(dst):
                src_path = os.path.join(candidate, item)
                (shutil.copytree if os.path.isdir(src_path) else shutil.copy2)(src_path, dst)
                restored += 1
        break
print(f'Restored {restored} item(s) from a previous session (0 is normal if this is a continuing session).')

def ckpt_path(name): return os.path.join(CHECKPOINT_DIR, name + '.pkl')
def ckpt_exists(name): return os.path.exists(ckpt_path(name))
def ckpt_save(name, obj):
    tmp = ckpt_path(name) + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(obj, f)
    os.replace(tmp, ckpt_path(name))
def ckpt_load(name):
    with open(ckpt_path(name), 'rb') as f: return pickle.load(f)

STOPWORDS = set(stopwords.words('english'))
FUNCTION_WORDS = ['the', 'of', 'and', 'to', 'in', 'is', 'that', 'it']
SEEDS = [42, 43, 44, 45, 46]


## 2. Load stage-2 artifacts (splits, features) — everything already computed, reused as-is

In [ ]:
def find_artifact(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

stage2_path = find_artifact(f'{ARTIFACTS_DIR}/m4_stage2.pkl', '/kaggle/working/m4_stage2.pkl',
                             *glob.glob('/kaggle/input/*/m4_stage2.pkl'), *glob.glob('/kaggle/input/*/*/m4_stage2.pkl'))
print('Loading stage-2 artifact from:', stage2_path)
with open(stage2_path, 'rb') as f:
    stage2 = pickle.load(f)

splits = stage2['splits']
Xs = stage2['Xs']            # flat scaled stylometric features, keys: train/val/testA/testB/testC
Xcat = stage2['Xcat']        # category-organized, same keys
cat_cols = stage2['cat_cols']
category_dims = stage2['category_dims']
feature_columns = stage2['feature_columns']

df_train, df_val = splits['train'], splits['val']
y_train, y_val = df_train.label.values, df_val.label.values
test_names = ['testA', 'testB', 'testC']
y_test = {name: splits[name].label.values for name in test_names}
print('Loaded splits:', {k: len(v) for k, v in splits.items()})


## 3. Build `val_compound` — reddit domain + dolly generator, disjoint from every existing split

Reloads the raw M4 files (needed to access the reddit+dolly pool, which isn't part of
any already-saved split) and explicitly checks disjointness against `val` before use.

In [ ]:
if not os.path.exists('/kaggle/working/M4'):
    !git clone -q https://github.com/mbzuai-nlp/M4.git /kaggle/working/M4
M4_ROOT = '/kaggle/working/M4/data'
MIN_WORDS = 30

def _wc(t): return len(t.split())
def clean(t): return re.sub(r'\s+', ' ', str(t)).strip()

def load_standard(path, domain, generator):
    human_rows, ai_rows = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: r = json.loads(line)
            except json.JSONDecodeError: continue
            ht, mt = clean(r.get('human_text', '')), clean(r.get('machine_text', ''))
            if _wc(ht) >= MIN_WORDS: human_rows.append({'text': ht, 'domain': domain, 'generator': 'human'})
            if _wc(mt) >= MIN_WORDS: ai_rows.append({'text': mt, 'domain': domain, 'generator': generator})
    return human_rows, ai_rows

reddit_human_raw, reddit_dolly_raw = [], []
for gen, fname in [('chatGPT','reddit_chatGPT.jsonl'), ('dolly','reddit_dolly.jsonl')]:
    path = os.path.join(M4_ROOT, fname)
    h, a = load_standard(path, 'reddit', gen)
    if gen == 'chatGPT': reddit_human_raw = h   # human text is identical across generator files for the same domain
    if gen == 'dolly': reddit_dolly_raw = a

df_reddit_human = pd.DataFrame(reddit_human_raw).drop_duplicates(subset=['text'])
df_reddit_dolly = pd.DataFrame(reddit_dolly_raw).drop_duplicates(subset=['text'])
df_reddit_human['label'] = 0
df_reddit_dolly['label'] = 1

VC_SEED = 43   # deliberately different from the M4_SEED=42 used for the original splits
N_VC = 300

used_val_human = set(df_val[df_val.label==0]['text'])
vc_human_pool = df_reddit_human[~df_reddit_human.text.isin(used_val_human)]
assert len(vc_human_pool) >= N_VC, f'Not enough spare reddit human docs: {len(vc_human_pool)}'
vc_human = vc_human_pool.sample(n=N_VC, random_state=VC_SEED)

assert len(df_reddit_dolly) >= N_VC, f'Not enough reddit+dolly docs: {len(df_reddit_dolly)}'
vc_ai = df_reddit_dolly.sample(n=N_VC, random_state=VC_SEED)

df_val_compound = pd.concat([vc_human, vc_ai], ignore_index=True).sample(frac=1, random_state=VC_SEED).reset_index(drop=True)
y_val_compound = df_val_compound.label.values

# Disjointness check against every existing split, not just val
all_existing_text = set()
for name, d in {**splits}.items():
    all_existing_text |= set(d['text'])
overlap = set(df_val_compound['text']) & all_existing_text
assert len(overlap) == 0, f'LEAKAGE: {len(overlap)} val_compound texts already appear in an existing split'
print(f'val_compound: {len(df_val_compound)} samples (domain=reddit, generator=dolly for AI class), '
      f'confirmed disjoint from train/val/testA/testB/testC.')
print(df_val_compound.label.value_counts())


## 4. Stylometric features for `val_compound`

Refits the extractor on `df_train` (same texts, same deterministic fitting procedure
as the original run, so this reproduces the identical POS-bigram vocabulary and
Burrows'-Delta reference — verified: `Counter.most_common()` is deterministic given
identical input) rather than requiring the original fitted-object pickle, which wasn't
part of the saved artifacts.

In [ ]:
def safe_div(a,b): return a/b if b else 0.0
def lexical_features(tokens):
    n = len(tokens)
    if n == 0: return {'hapax_ratio':0.,'yules_k':0.,'ttr':0.,'avg_word_len':0.}
    freqs = Counter(tokens); V = len(freqs)
    hapax = sum(1 for w,c in freqs.items() if c==1)
    freq_of_freq = Counter(freqs.values())
    sum_i2fi = sum((i**2)*fi for i,fi in freq_of_freq.items())
    return {'hapax_ratio':hapax/n,'yules_k':1e4*(sum_i2fi-n)/(n**2),'ttr':V/n,
            'avg_word_len':float(np.mean([len(w) for w in tokens]))}
def syntactic_features(sentences):
    lens = [len(word_tokenize(s)) for s in sentences] if sentences else [0]
    mean_len, var_len = float(np.mean(lens)), float(np.var(lens))
    burstiness = (var_len-mean_len)/(var_len+mean_len) if (var_len+mean_len)>0 else 0.
    full_text = ' '.join(sentences); n_chars = max(len(full_text),1)
    n_punct = sum(1 for c in full_text if c in '.,;:!?')
    return {'sent_len_variance':var_len,'burstiness':burstiness,'punct_density':n_punct/n_chars,'avg_sent_len':mean_len}
def pos_bigrams(tagged):
    tags = [t for _,t in tagged]; return list(zip(tags, tags[1:]))
def fit_pos_bigram_vocab(train_texts, top_k=36):
    counter = Counter()
    for text in tqdm(train_texts, desc='Refitting POS-bigram vocab (train only, deterministic)'):
        counter.update(pos_bigrams(pos_tag(word_tokenize(text))))
    return [bg for bg,_ in counter.most_common(top_k)]
def grammatical_features(tagged, vocab):
    bigrams = pos_bigrams(tagged); total = len(bigrams); counts = Counter(bigrams)
    return {f'pos_{a}_{b}': (counts.get((a,b),0)/total if total>0 else 0.) for a,b in vocab}
def build_burrows_reference(train_human_texts, top_words):
    rates = {w: [] for w in top_words}
    for text in train_human_texts:
        tokens = [t.lower() for t in word_tokenize(text)]; n = max(len(tokens),1); freqs = Counter(tokens)
        for w in top_words: rates[w].append(freqs.get(w,0)/n)
    return ({w: float(np.mean(v)) for w,v in rates.items()}, {w: (float(np.std(v)) if np.std(v)>0 else 1.0) for w,v in rates.items()})
def burrows_delta(tokens, ref_mean, ref_std, top_words):
    n = max(len(tokens),1); freqs = Counter(tokens)
    diffs = [abs(((freqs.get(w,0)/n)-ref_mean.get(w,0.))/ref_std.get(w,1.)) for w in top_words]
    return float(np.mean(diffs)) if diffs else 0.
def function_word_ratios(tokens):
    n = max(len(tokens),1); freqs = Counter(t.lower() for t in tokens)
    return {f'func_{w}': freqs.get(w,0)/n for w in FUNCTION_WORDS}

class StylometricExtractor:
    def fit(self, train_texts, train_human_texts, n_bigrams=36, n_burrows_words=20):
        self.bigram_vocab = fit_pos_bigram_vocab(train_texts, n_bigrams)
        all_tok = [w.lower() for t in train_human_texts for w in word_tokenize(t)]
        self.burrows_words = [w for w,_ in Counter(all_tok).most_common(n_burrows_words) if w.isalpha()]
        self.ref_mean, self.ref_std = build_burrows_reference(train_human_texts, self.burrows_words)
        return self
    def transform(self, text, gpt2_ppl_fn=None):
        tokens, sentences = word_tokenize(text), sent_tokenize(text)
        tagged = pos_tag(tokens)
        feats = {}
        feats.update(lexical_features([t.lower() for t in tokens]))
        feats.update(syntactic_features(sentences))
        feats.update(grammatical_features(tagged, self.bigram_vocab))
        feats['burrows_delta'] = burrows_delta([t.lower() for t in tokens], self.ref_mean, self.ref_std, self.burrows_words)
        feats['gpt2_perplexity'] = gpt2_ppl_fn(text) if gpt2_ppl_fn else np.nan
        feats.update(function_word_ratios(tokens))
        return feats
    def transform_batch(self, texts, gpt2_ppl_fn=None, desc='Extracting'):
        rows = [self.transform(t, gpt2_ppl_fn) for t in tqdm(texts, desc=desc)]
        return pd.DataFrame(rows)

extractor_ckpt = 'm4_stylometric_extractor_refit'
if ckpt_exists(extractor_ckpt):
    extractor = ckpt_load(extractor_ckpt)
else:
    extractor = StylometricExtractor().fit(df_train['text'].tolist(), df_train[df_train.label==0]['text'].tolist())
    ckpt_save(extractor_ckpt, extractor)

# Sanity check: refit vocabulary must exactly match the columns already in the loaded Xs/feature_columns
assert [f'pos_{a}_{b}' for a,b in extractor.bigram_vocab] == [c for c in feature_columns if c.startswith('pos_')], \
    'Refit POS-bigram vocab does not match the original -- do not proceed, features would be inconsistent'
print('Refit stylometric extractor matches the original vocabulary exactly -- safe to proceed.')

_gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
_gpt2_lm = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()

@torch.no_grad()
def gpt2_perplexity(text, max_len=512):
    ids = _gpt2_tok(text, return_tensors='pt', truncation=True, max_length=max_len).input_ids.to(DEVICE)
    if ids.shape[1] < 2: return float('nan')
    loss = _gpt2_lm(ids, labels=ids).loss
    return float(torch.exp(loss).item())

vc_style_ckpt = 'm4_val_compound_style'
if ckpt_exists(vc_style_ckpt):
    X_style_vc = ckpt_load(vc_style_ckpt)
else:
    X_style_vc = extractor.transform_batch(df_val_compound['text'].tolist(), gpt2_perplexity, 'val_compound style')
    ckpt_save(vc_style_ckpt, X_style_vc)

train_ppl_median = None  # recomputed below from the original Xs -- val_compound NaNs filled consistently
X_style_vc['gpt2_perplexity'] = X_style_vc['gpt2_perplexity'].fillna(X_style_vc['gpt2_perplexity'].median())
print('val_compound stylometric features:', X_style_vc.shape)


## 5. Frozen BERT embeddings for `val_compound` (small, ~600 samples)

Train and test-set embeddings are reused from the stage-2 artifacts / prior `.npy`
caches — only `val_compound`'s embeddings are new.

In [ ]:
_bert_tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
_bert_model = AutoModel.from_pretrained('distilbert-base-uncased').to(DEVICE).eval()

@torch.no_grad()
def get_cls_embeddings(texts, batch_size=32, max_length=512, desc='BERT embeddings'):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        enc = _bert_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
        out = _bert_model(**enc).last_hidden_state[:, 0, :]
        embs.append(out.cpu().numpy())
    return np.vstack(embs)

def load_bert_npy(name):
    for path in [f'{ARTIFACTS_DIR}/{name}.npy', f'/kaggle/working/{name}.npy',
                 *glob.glob(f'/kaggle/input/*/{name}.npy'), *glob.glob(f'/kaggle/input/*/*/{name}.npy')]:
        if os.path.exists(path):
            return np.load(path)
    return None

Xb_train = load_bert_npy('m4_bert_train')
if Xb_train is None:
    print('m4_bert_train.npy not found in any known location -- recomputing (one-time cost).')
    Xb_train = get_cls_embeddings(df_train['text'].tolist(), desc='BERT: train (recompute)')
    np.save(f'{ARTIFACTS_DIR}/m4_bert_train.npy', Xb_train)

Xb_test = {}
for name in test_names:
    arr = load_bert_npy(f'm4_bert_{name}')
    if arr is None:
        print(f'm4_bert_{name}.npy not found -- recomputing.')
        arr = get_cls_embeddings(splits[name]['text'].tolist(), desc=f'BERT: {name} (recompute)')
        np.save(f'{ARTIFACTS_DIR}/m4_bert_{name}.npy', arr)
    Xb_test[name] = arr

vc_bert_ckpt = 'm4_val_compound_bert'
if ckpt_exists(vc_bert_ckpt):
    Xb_val_compound = ckpt_load(vc_bert_ckpt)
else:
    Xb_val_compound = get_cls_embeddings(df_val_compound['text'].tolist(), desc='BERT: val_compound')
    ckpt_save(vc_bert_ckpt, Xb_val_compound)

print('Xb_train:', Xb_train.shape, '| Xb_val_compound:', Xb_val_compound.shape,
      '| Xb_test shapes:', {k: v.shape for k, v in Xb_test.items()})


## 6. Attention-gated model + trainer — identical architecture, only the checkpoint-selection data changes

In [ ]:
class AttentionGatedHybridDetector(nn.Module):
    def __init__(self, category_dims, bert_dim=768, cat_hidden=32, ffn_out=64):
        super().__init__()
        self.category_names = list(category_dims.keys())
        self.category_encoders = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(dim, cat_hidden), nn.ReLU(), nn.Dropout(0.2))
            for name, dim in category_dims.items()})
        n_cat = len(category_dims)
        self.gate = nn.Linear(cat_hidden * n_cat, n_cat)
        self.style_proj = nn.Sequential(nn.Linear(cat_hidden, ffn_out), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(bert_dim + ffn_out, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
    def forward(self, bert_emb, category_feats, return_gate=False):
        encoded = [self.category_encoders[name](category_feats[name]) for name in self.category_names]
        concat_encoded = torch.cat(encoded, dim=1)
        gate_weights = torch.softmax(self.gate(concat_encoded), dim=1)
        stacked = torch.stack(encoded, dim=1)
        weighted = (stacked * gate_weights.unsqueeze(-1)).sum(dim=1)
        style_repr = self.style_proj(weighted)
        logits = self.classifier(torch.cat([bert_emb, style_repr], dim=1))
        return (logits, gate_weights) if return_gate else logits

class CategoryDataset(Dataset):
    def __init__(self, bert_emb, cat_dict, labels):
        self.bert_emb = torch.tensor(bert_emb, dtype=torch.float32)
        self.cat_dict = {k: torch.tensor(v, dtype=torch.float32) for k, v in cat_dict.items()}
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.bert_emb[i], {k: v[i] for k, v in self.cat_dict.items()}, self.labels[i]

def collate_cat(batch):
    bert_embs = torch.stack([b[0] for b in batch])
    cat_names = batch[0][1].keys()
    cats = {name: torch.stack([b[1][name] for b in batch]) for name in cat_names}
    labels = torch.stack([b[2] for b in batch])
    return bert_embs, cats, labels

def make_cat_loader(bert_emb, cat_dict, labels, batch_size=64, shuffle=True):
    return DataLoader(CategoryDataset(bert_emb, cat_dict, labels), batch_size=batch_size, shuffle=shuffle, collate_fn=collate_cat)

def evaluate_gated(model, loader, threshold=0.5):
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for be, cats, lb in loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}
            probs = torch.softmax(model(be, cats), dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy()); all_labels.extend(lb.numpy())
    all_labels, all_probs = np.array(all_labels), np.array(all_probs)
    return all_labels, (all_probs >= threshold).astype(int), all_probs

def calibrate_threshold(labels, probs, n_steps=199):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.01, 0.99, n_steps):
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    return best_t, best_f1

def train_gated(train_loader, selection_loader, category_dims, seed, epochs=15, lr=1e-3, weight_decay=0.01, patience=5):
    """`selection_loader` is whatever data checkpoint selection + threshold calibration
    uses -- the ONLY thing that changes between the original run and this experiment."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AttentionGatedHybridDetector(category_dims).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = get_cosine_schedule_with_warmup(opt, int(0.1*len(train_loader)*epochs), len(train_loader)*epochs)
    criterion = nn.CrossEntropyLoss()
    best_val_f1, best_state, epochs_no_improve = -1, None, 0
    for ep in range(1, epochs+1):
        model.train()
        for be, cats, lb in train_loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}; lb = lb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(be, cats), lb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
        sel_labels, sel_preds, _ = evaluate_gated(model, selection_loader)
        sel_f1 = f1_score(sel_labels, sel_preds, zero_division=0)
        if sel_f1 > best_val_f1:
            best_val_f1, best_state, epochs_no_improve = sel_f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience: break
    model.load_state_dict(best_state)
    sel_labels, _, sel_probs = evaluate_gated(model, selection_loader)
    calibrated_t, _ = calibrate_threshold(sel_labels, sel_probs)
    return model, best_val_f1, calibrated_t

def full_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0),
            'fpr': fp/(fp+tn) if (fp+tn) > 0 else 0.0}


## 7. Retrain with compound-validation checkpoint selection — 5 seeds

Same train data, same architecture, same seeds as the original run. The only variable
changed is what `train_gated` uses for early stopping and threshold calibration:
`val_compound` (reddit+dolly) instead of `val` (reddit, train generators).

In [ ]:
# Scale val_compound's stylometric features using the ORIGINAL train-fitted scaler parameters.
# The scaler itself isn't in the pickle, but Xs['train'] (already scaled) plus the raw train features
# let us refit an identical StandardScaler deterministically (same data, same fit).
#
# CRITICAL: this cell must call style_scaler.transform() on the validation features before use.
# An earlier hand-edited version of this experiment accidentally skipped that call (used raw,
# unstandardized features directly), which silently corrupted checkpoint selection -- the model's
# category encoders expect roughly [-3, 3]-range inputs, and raw features can range into the
# hundreds. Both an equality check (against the scaler) and a range check (against the output)
# are included below so that mistake cannot silently repeat.
from sklearn.preprocessing import StandardScaler
X_style_train_raw_ckpt = 'm4_style_train_raw_refit'
if ckpt_exists(X_style_train_raw_ckpt):
    X_style_train_raw = ckpt_load(X_style_train_raw_ckpt)
else:
    X_style_train_raw = extractor.transform_batch(df_train['text'].tolist(), gpt2_perplexity, 'refit train style (for scaler)')
    ckpt_save(X_style_train_raw_ckpt, X_style_train_raw)
X_style_train_raw['gpt2_perplexity'] = X_style_train_raw['gpt2_perplexity'].fillna(X_style_train_raw['gpt2_perplexity'].median())

style_scaler = StandardScaler().fit(X_style_train_raw.values)
# Check 1: refit-scaled train features should closely match the already-scaled Xs['train'] from stage2
refit_scaled_train = style_scaler.transform(X_style_train_raw.values)
max_abs_diff = np.abs(refit_scaled_train - Xs['train']).max()
print(f'Max abs difference between refit-scaled and original Xs["train"]: {max_abs_diff:.6f} (should be ~0)')
assert max_abs_diff < 1e-6, 'Refit scaler does not match original -- do not proceed'

Xs_val_compound = style_scaler.transform(X_style_vc.values)

# Check 2: the SCALED val_compound features must land in a standardized range, not a raw one.
# This is the specific check that would have caught the earlier bug immediately.
print(f'Scaled val_compound feature range: [{Xs_val_compound.min():.2f}, {Xs_val_compound.max():.2f}] (expect roughly [-10, 10])')
assert abs(Xs_val_compound.min()) < 20 and abs(Xs_val_compound.max()) < 20, (
    'val_compound features are NOT on a standardized scale -- style_scaler.transform() was likely '
    'skipped. Do not proceed to training with this data; re-check this cell before continuing.'
)
print('Scale check passed.')

col_index = {c: i for i, c in enumerate(feature_columns)}
Xcat_val_compound = {cat: Xs_val_compound[:, [col_index[c] for c in cols]] for cat, cols in cat_cols.items()}
print('val_compound category-organized feature shapes:', {k: v.shape for k, v in Xcat_val_compound.items()})


In [ ]:
train_loader = make_cat_loader(Xb_train, Xcat['train'], y_train, batch_size=64, shuffle=True)
selection_loader = make_cat_loader(Xb_val_compound, Xcat_val_compound, y_val_compound, batch_size=64, shuffle=False)
test_loaders = {name: make_cat_loader(Xb_test[name], Xcat[name], y_test[name], batch_size=128, shuffle=False) for name in test_names}

hybrid_v2_results = {name: [] for name in test_names}
hybrid_v2_raw = {name: [] for name in test_names}
gate_v2_log = []

for seed in SEEDS:
    ckpt_name = f'm4_hybrid_v2_compoundval_seed{seed}'
    if ckpt_exists(ckpt_name):
        seed_result = ckpt_load(ckpt_name)
    else:
        model, best_sel_f1, calibrated_t = train_gated(train_loader, selection_loader, category_dims, seed=seed)
        seed_result = {'metrics': {}, 'raw': {}, 'gate': None, 'sel_f1': best_sel_f1, 'threshold': calibrated_t}
        for name in test_names:
            labels, preds, probs = evaluate_gated(model, test_loaders[name], threshold=calibrated_t)
            m = full_metrics(labels, preds); m['seed'] = seed; m['threshold'] = calibrated_t
            seed_result['metrics'][name] = m
            seed_result['raw'][name] = (labels, preds)
        with torch.no_grad():
            gates = []
            for be, cats, lb in test_loaders[test_names[0]]:
                be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}
                _, gw = model(be, cats, return_gate=True); gates.append(gw.cpu().numpy())
            seed_result['gate'] = dict(zip(category_dims.keys(), np.concatenate(gates).mean(axis=0).tolist()))
        ckpt_save(ckpt_name, seed_result)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    for name in test_names:
        hybrid_v2_results[name].append(seed_result['metrics'][name])
        hybrid_v2_raw[name].append(seed_result['raw'][name])
    gate_v2_log.append(seed_result['gate'] | {'seed': seed})
    print(f'seed {seed}: compound-val F1={seed_result["sel_f1"]:.4f}, threshold={seed_result["threshold"]:.3f}, ' +
          ', '.join(f'{n}_f1={seed_result["metrics"][n]["f1"]:.4f}' for n in test_names))


## 8. Compare against the original (reddit-val-selected) Hybrid results

In [ ]:
def load_original(name):
    for path in [f'{ARTIFACTS_DIR}/M4_hybrid_{name}.csv', f'/kaggle/working/M4_hybrid_{name}.csv',
                 *glob.glob(f'/kaggle/input/*/M4_hybrid_{name}.csv'), *glob.glob(f'/kaggle/input/*/*/M4_hybrid_{name}.csv')]:
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f'Original M4_hybrid_{name}.csv not found -- upload it alongside this notebook\'s inputs.')

comparison_rows = []
mcnemar_rows = []
for name in test_names:
    orig = load_original(name)
    new_df = pd.DataFrame(hybrid_v2_results[name])
    comparison_rows.append({'test_set': name, 'variant': 'Original (reddit-val)',
                             'f1_mean': orig.f1.mean(), 'f1_std': orig.f1.std(),
                             'acc_mean': orig.accuracy.mean(), 'fpr_mean': orig.fpr.mean()})
    comparison_rows.append({'test_set': name, 'variant': 'Compound-val (reddit+dolly selection)',
                             'f1_mean': new_df.f1.mean(), 'f1_std': new_df.f1.std(),
                             'acc_mean': new_df.accuracy.mean(), 'fpr_mean': new_df.fpr.mean()})


comp_df = pd.DataFrame(comparison_rows).round(4)
print(comp_df.to_string(index=False))
comp_df.to_csv(f'{ARTIFACTS_DIR}/M4_compound_val_comparison.csv', index=False)
print()
print('NOTE: the original run only saved aggregate metrics (accuracy/f1/fpr per seed), not raw')
print('per-sample predictions -- confirmed by checking M4_hybrid_testC.csv columns. This means a')
print('paired McNemar test against the EXACT original run is not possible from what is available;')
print('the comparison above is mean-vs-mean across 5 seeds each, which is still informative but')
print('weaker than a paired test. Bootstrap CIs below characterize the NEW runs uncertainty directly.')

def bootstrap_ci_f1(labels, preds, n_boot=3000, seed=42):
    rng = np.random.default_rng(seed); labels, preds = np.asarray(labels), np.asarray(preds); n = len(labels)
    stats = [f1_score(labels[idx], preds[idx], zero_division=0) for idx in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(np.mean(stats)), float(lo), float(hi)

ci_rows = []
for name in test_names:
    for seed_i, (labels, preds) in enumerate(hybrid_v2_raw[name]):
        mean_f1, lo, hi = bootstrap_ci_f1(labels, preds, seed=SEEDS[seed_i])
        ci_rows.append({'test_set': name, 'seed': SEEDS[seed_i], 'f1_mean': mean_f1, 'ci_lower': lo, 'ci_upper': hi})
ci_df = pd.DataFrame(ci_rows).round(4)
print(ci_df.to_string(index=False))
ci_df.to_csv(f'{ARTIFACTS_DIR}/M4_compound_val_bootstrap_ci.csv', index=False)

gate_v2_df = pd.DataFrame(gate_v2_log)
print()
print('=== Attention gate weights: compound-val-selected (compare against the original run\'s near-uniform weights) ===')
print(gate_v2_df.to_string(index=False))
gate_v2_df.to_csv(f'{ARTIFACTS_DIR}/M4_compound_val_gates.csv', index=False)

print()
print('If grammatical weight is now consistently higher than in the original run (0.19-0.32 range, no clear preference),')
print('and/or testC F1 mean improved meaningfully above 0.1965, the compound-validation fix shows a real effect.')
print('If gate weights and testC F1 look similar to before, the fix did not change what the model learned,')
print('and the compound-shift failure is likely a deeper capacity/data limitation, not a checkpoint-selection artifact.')
